In [1]:
# Cell 1: Environment Setup

include("scripts/initialization_helpers.jl")
include("scripts/initialization_dictionaries.jl")
using .InitializationHelpers
using .InitializationDictionaries
using OMJulia
using Plots, DataFrames, CSV

# --- Configuration ---

# 1. Directory containing the model files
MODEL_DIR = abspath("models")

# 2. Select the model to initialize
MODEL = "MyPVCurrent"

# 3. Path to the selected dynamic case file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Name and path of the initialized dynamic case file
INITIALIZED_MODEL = MODEL * "_initialized"
INITIALIZED_FILE_PATH = joinpath(MODEL_DIR, INITIALIZED_MODEL * ".mo")

# 5. Path to the auxiliary file
AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE_PATH = joinpath(MODEL_DIR, AUX_MODEL * ".mo")

# 6. Path to the Dynawo package.mo
DYNAWO_PKG_PATH = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 7. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 8. Variable to plot after the initialized simulation
# PLOT_VARIABLE = "generatorSynchronous.terminal.i.re"  # SMIB option
# PLOT_VARIABLE = "BESS.terminal.V.im"  # MyBESS option
PLOT_VARIABLE = "PV.measurements.PPu"


"BESS.measurements.PPu"

In [2]:
# Cell 2: OpenModelica Setup + Model Loading

# 1. Load the dynamic model
BESS = OMJulia.OMCSession()
sendExpression(BESS, "loadModel(Complex)")
sendExpression(BESS, "loadModel(ModelicaServices)")
sendExpression(BESS, "loadFile(\"$MODELICA_PKG_PATH\")")
sendExpression(BESS, "loadFile(\"$DYNAWO_PKG_PATH\")")
sendExpression(BESS, "loadFile(\"$MODEL_FILE_PATH\")")
sendExpression(BESS, "clearMessages()")
println("Checking the dynamic model...")
chk_dyn = sendExpression(BESS, "checkModel($MODEL)", parsed=false)
println(chk_dyn)

# 2. Load the auxiliary model
StaticBESS = OMJulia.OMCSession()
sendExpression(StaticBESS, "loadModel(Complex)")
sendExpression(StaticBESS, "loadModel(ModelicaServices)")
sendExpression(StaticBESS, "loadFile(\"$MODELICA_PKG_PATH\")")
sendExpression(StaticBESS, "loadFile(\"$DYNAWO_PKG_PATH\")")
sendExpression(StaticBESS, "loadFile(\"$AUX_FILE_PATH\")")
sendExpression(StaticBESS, "clearMessages()")
println("Checking the auxiliary model...")
chk_aux = sendExpression(StaticBESS, "checkModel($AUX_MODEL)", parsed=false)
println(chk_aux)


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.X0CmiMvTiG"


Checking the dynamic model...
"Check of MyPVCurrent completed successfully.
Class MyPVCurrent has 537 equation(s) and 537 variable(s).
374 of these are trivial equation(s)."



[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.KrzIXo6pha"


Checking the auxiliary model...
"Check of MyPVCurrent_auxiliary completed successfully.
Class MyPVCurrent_auxiliary has 61 equation(s) and 61 variable(s).
17 of these are trivial equation(s)."



In [3]:
# Cell 3: Simulate the auxiliary model and extract initialization values
# 1. Build and simulate the auxiliary model
ModelicaSystem(StaticBESS, AUX_FILE_PATH, AUX_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
simulate(StaticBESS, resultfile = AUX_MODEL * "_res.mat")

# 2. Get the dynamic model components that need initialization
components = get_all_components(BESS, MODEL)
initializable_components = get_initializable_components(components, INIT_PARAMS)

# 3. Extract initialization values from the auxiliary model
init_values_by_component = extract_all_initialization_values(StaticBESS, initializable_components, INIT_PARAMS)


Dict{String, Dict{String, Float64}} with 1 entry:
  "PV" => Dict("PF0"=>0.994533, "QInj0Pu"=>0.0734995, "Q0Pu"=>5.06203e-7, "i0Pu…

In [4]:
# Cell 4: Build and save the initialized dynamic model
om_send(BESS, "deleteClass($INITIALIZED_MODEL)")
om_send(BESS, "clearMessages()")
om_send(BESS, "copyClass($MODEL, \"$INITIALIZED_MODEL\")")

apply_initialization_modifiers!(BESS, INITIALIZED_MODEL, initializable_components, INIT_PARAMS, init_values_by_component)

om_send(BESS, "saveModel(\"$INITIALIZED_FILE_PATH\", $INITIALIZED_MODEL)")
println("Wrote initialized model: ", INITIALIZED_FILE_PATH)


OMC -> deleteClass(MyPVCurrent_initialized)
OMC -> clearMessages()
OMC -> copyClass(MyPVCurrent, "MyPVCurrent_initialized")
OMC -> updateComponent(PV, Dynawo.Electrical.Photovoltaics.WECC.PVCurrentSource, MyPVCurrent_initialized, modification = $Code((u0Pu = Complex(0.9999999999989542, -1.4462699999999998e-6), KiPLL = 20, DDn = 20, Kqi = 0.5, U0Pu = 1, tFv = 0.1, Dbd2Pu = 0.1, Q0Pu = 5.062033898185934e-7, EMaxPu = 999, UInj0Pu = 1.005497313839768, VRef0Pu = 1, tG = 0.02, Kvp = 1, Id0Pu = 0.6961729189776327, VMaxPu = 1.1, QMaxPu = 0.4, VUpPu = 1.1, Kc = 0, Kqv = 2, IMaxPu = 1.05, QFlag = true, VFrz = 0, RefFlag = true, tFilterPC = 0.04, i0Pu = Complex(-0.7, 5.061856101819357e-7), s0Pu = Complex(-0.7, 5.062033898185934e-7), QInj0Pu = 0.07349949379664861, VFlag = true, Ki = 1.5, PfFlag = false, brkpt = 0.1, EMinPu = -999, IqrMinPu = -20, RPu = 0, lvpl1 = 1.22, P0Pu = -0.7, QMinPu = -0.4, PMinPu = 0, DPMaxPu = 999, UPhase0 = 1.44621e-6, DbdPu = 0.01, FEMaxPu = 999, tFilterGC = 0.02, tIq = 

In [5]:
# Cell 5: Final simulation
InitializedBESS = OMJulia.OMCSession()
sendExpression(InitializedBESS, "loadModel(Complex)")
sendExpression(InitializedBESS, "loadModel(ModelicaServices)")
ModelicaSystem(InitializedBESS, INITIALIZED_FILE_PATH, INITIALIZED_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
sendExpression(InitializedBESS, "clearMessages()")
println("Checking the initialized model...")
chk_init = sendExpression(InitializedBESS, "checkModel($INITIALIZED_MODEL)", parsed=false)
println(chk_init)

initialized_resultfile_name = INITIALIZED_MODEL * ".csv"
simulate(InitializedBESS, resultfile = initialized_resultfile_name, simflags = "-outputFormat=csv")
initialized_resultfile = joinpath(getWorkDirectory(InitializedBESS), initialized_resultfile_name)


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.4v0M5wRkdg"


Checking the initialized model...
"Check of MyPVCurrent_initialized completed successfully.
Class MyPVCurrent_initialized has 537 equation(s) and 537 variable(s).
374 of these are trivial equation(s)."



"/tmp/jl_uK5QuS/MyPVCurrent_initialized.csv"

In [6]:
# Cell 6: Plot the initialized dynamic model
initialized_df = DataFrame(CSV.File(initialized_resultfile))

plotlyjs()
p = plot(initialized_df[!, "time"], initialized_df[!, PLOT_VARIABLE], label = [PLOT_VARIABLE])
plot!(p, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p, "Initialized dynamic model response")
xlabel!(p, "Time (s)")
ylabel!(p, PLOT_VARIABLE)


WebIO._IJuliaInit()

LoadError: ArgumentError: column name :BESS.measurements.PPu not found in the data frame